Da, apsolutno bih to dodao, ali pametno i kontrolisano. To ti je baš sljedeći nivo ozbiljnosti.

Moj redoslijed bi bio:

1. Prvo Optuna za LGBMRanker
2. Onda XGBRanker kao challenger model
3. Ne GridSearchCV kao primarni pristup
Zašto ne bih prvo GridSearchCV: kod ranking modela imaš group parametar, temporal splitove i custom metrike. GridSearchCV je super za standardne sklearn modele, ali ovdje lako postane nezgrapan i spor. Scikit-learn dokumentacija i sama razlikuje exhaustive GridSearchCV i RandomizedSearchCV, gdje grid iscrpno prolazi sve kombinacije. Za tvoj slučaj Optuna je praktičnija jer možeš direktno optimizovati blend_hr3 ili blend_mrr5.

XGBRanker ima smisla probati. XGBoost dokumentacija podržava ranking kroz sklearn-style XGBRanker, ali kod ranking modela treba paziti na grupe/query strukturu. Znači nije “drop-in” bez provjere, ali je vrlo dobar challenger.

Šta bih optimizovao
Ne bih optimizovao samo model HR@3. Pošto ti blend radi bolje, optimizovao bih:

primary: blend_hr3
secondary: blend_mrr5 ili blend_hr5
I težinu blenda bih stavio kao hyperparametar:

blend_model_weight = trial.suggest_float("blend_model_weight", 0.1, 0.9)
blend_markov_weight = 1 - blend_model_weight
Možda Optuna sama potvrdi da je 0.2-0.4 model / 0.6-0.8 Markov najbolji.

Minimalni Optuna objective za LGBMRanker

Ovo bih dodao kao posebnu sekciju nakon što ti pipeline stabilno pravi train_dataset, val_dataset, X_train, X_val, y_train, y_val, group_train, group_val.

In [ ]:
import optuna
from lightgbm import LGBMRanker

def objective_lgbm(trial):
    params = {
        "objective": "lambdarank",
        "n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 8, 64),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 80),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 5.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42,
        "verbosity": -1,
    }

    blend_model_weight = trial.suggest_float("blend_model_weight", 0.1, 0.9)

    model = LGBMRanker(**params)
    model.fit(
        X_train, y_train,
        group=group_train,
        eval_set=[(X_val, y_val)],
        eval_group=[group_val],
        eval_at=[1, 3, 5],
        callbacks=[early_stopping(stopping_rounds=50, verbose=False)]
    )

    preds = model.predict(X_val)

    model_score_norm = normalize_by_group(preds, group_val)
    markov_score_norm = normalize_by_group(val_dataset["markov_prob"].values, group_val)

    blended_score = (
        blend_model_weight * model_score_norm +
        (1 - blend_model_weight) * markov_score_norm
    )

    return hit_rate_at_k(blended_score, y_val.values, group_val, 3)

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective_lgbm, n_trials=50)

print("Best value:", study.best_value)
print("Best params:", study.best_params)

In [ ]:
from xgboost import XGBRanker

In [ ]:
xgb_model = XGBRanker(
    objective="rank:ndcg",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=42,
)

Fit je sličan, ali obavezno testirati da tvoja verzija XGBoost-a prima group ovako:

xgb_model.fit(
    X_train,
    y_train,
    group=group_train,
    eval_set=[(X_val, y_val)],
    eval_group=[group_val],
    verbose=False
)
Ako prođe, isto računaš:

preds_xgb = xgb_model.predict(X_val)
pa isti blend:

model_score_norm = normalize_by_group(preds_xgb, group_val)
markov_score_norm = normalize_by_group(val_dataset["markov_prob"].values, group_val)

blended_xgb = 0.3 * model_score_norm + 0.7 * markov_score_norm

Kako bih to prodao
Ovo bih nazvao:

Model benchmark:
- Markov baseline
- LightGBM Ranker
- LightGBM + Optuna
- XGBoost Ranker
- XGBoost + Optuna
- Best blend ranking
To izgleda mnogo ozbiljnije pred klijentom jer pokazuje da nisi slučajno izabrao jedan model, nego si radio model selection.

Moj savjet: da, dodaj Optuna i XGBRanker, ali nemoj sad širiti na 10 modela. Dovoljno je:

Baseline Markov
LGBMRanker tuned
XGBRanker tuned
Best blend
To je profesionalno, a i dalje održivo.